# Python中級編4: ファイルの読み書き（1）


## 目次
* [ファイルとは何か](#ファイルとは何か)
* [テキストファイルと文字コード](#テキストファイルと文字コード)
  * [ミニ練習: バイト列が表す文字](#ミニ練習:-バイト列が表す文字)
  * [ミニ練習：文字を表すバイト列](#ミニ練習：文字を表すバイト列)
* [テキストファイルの読み込み](#テキストファイルの読み込み)
 * [ファイルの open](#ファイルの-open)
 * [1行読み込む（readline）](#1行読み込む（readline）)
 * [ファイルの close](#ファイルの-close)
 * [1行づつ読んでファイル全体を処理するパターン](#1行づつ読んでファイル全体を処理するパターン)
 * [練習13.1: ファイルの行数を数える](#練習13.1:-ファイルの行数を数える)
 * [練習13.2: 特定の文字列を含む行数を数える](#練習13.2:-特定の文字列を含む行数を数える)
* [テキストファイルの書き込み](#テキストファイルの書き込み)
 * [練習13.3: 先頭行の書き出し](#練習13.3:-先頭行の書き出し)
 * [練習13.4: キーワードを含む行の書き出し](#練習13.4:-キーワードを含む行の書き出し)

---
**始める前の注意**：

今回のノートブックは
```python
!ls -l 
```
のようなターミナルコマンド（シェルコマンド）を実行するセルが沢山あります。
ターミナルコマンドとは、上の例では `ls -l` の部分で、その名の通りターミナル上で打ち込めば実行できるコマンド（小さなプログラム）です。
Jupyter Notebook の中からターミナルコマンドを実行する場合は、行の先頭に `!` を書き、そのあと続けてターミナルコマンドを書きます。

<font color="red">このノートブックの中で使っているターミナルコマンドは Mac 用です。普段は Windows で課題をやっている人も今回は Mac でやってください</font>

練習のいくつかは、ターミナルコマンドと同様の動作をするプログラムを書くという内容です。
そのような練習では、ターミナルコマンドを使うのではなく、Python のプログラムを書いてください。

---
**このノートブックの目的**

これまでの講義では，例えば
```
小説「吾輩は猫である」＋検索キーワードを入力 → [キーワード検索の処理] → キーワードを含む行を出力
```
あるいは
```
行列のデータを入力 →　[行列式の計算処理] →　行列式の値を出力
```
のように
```
データを入力 →　[データを処理をする] →　処理結果を出力
```
のというパターンのうち，主に `[データを処理する]` の部分の練習を行ってきた．

しかし，実際にプログラムを書いてコンピュータを使いこなすためには
```
データを入力 →
```
の部分と
```
→ 処理結果を出力
```
の部分の仕組みを少し理解する必要がある．

多くの場合，プログラムに入力されるデータや，プログラムから出力されるデータは
「ファイル」という形でコンピュータ内に保存されている．

つまり`入力 →　[計算処理] →　出力` というパターンは具体的には
1. コンピュータ内のファイルを読み込んでプログラムに入力する（ファイル入力）
2. 入力されたファイルについて計算処理する
3. 計算処理の結果をコンピュータ内に新しいファイルとして保存する（ファイル出力）

の３つのステップからなる。

このノートブックでは、「ファイル入力」と「ファイル出力」の部分まで含めたプログラミングの練習を行う。

始める前に、LETUS の今回のセクションから以下のファイルをダウンロードし、このノートブックと同じフォルダに置きなさい。
* neko-noseg.txt

## ファイルとは何か
コンピュータの中では全てのデータは 0 と 1 を並べた2進数で表される。
つまり、ひとかたまりのデータは
```
000101010101010101101111110101010111011011101010111011011010000101010101
```
のような大きな2進数と見ることができる。

しかし桁数が定まっていないデータは扱いづらいし、あまり大きな数をそのまま計算に使うことはできないので、通常は2進数のデータを8桁ごとに区切って考える：
```
00010101 01010101 01101111 11010101 01110110 11101010 11101101 10100001 01010101
```

これらの8桁の2進数それぞれを「バイト」という。

よって1バイトは`00000000` $= 0$ から`11111111` $= 2^8-1 = 255$ の範囲の整数を表す。

さらに1バイトを前半の4桁と後半の4桁に分け、それぞれに 0 から f の文字を割りあてた表記をよく用いる（つまり16進数としての表記）。

|2進表記  | 0000 | 0001 | 0010 | 0011 | ... | 1000 | 1001 | 1010 | 1011 | 1100 | 1101 | 1101 | 1111 |
|--------|:----:|:----:|:----:|:----:|:----:|:----:|:----:|:----:|:----:|:----:|:----:|:----:|:----:|
|10進表記 |   0  |   1  |   2  |   3  | ... |   8  |   9  |  10  |  11  |  12  |  13  |  14  | 15   |
|16進表記 |   0  |   1  |   2  |   3  | ... |   8  |   9  |   a  |   b  |   c  |   d  |   e  |   f  |

この16進表記では例えばバイト `e4` は $14 \times 16 + 4 = 228$ を表す。

ファイルとは、おおまかに言えば「コンピューターの電源をオフにしても消えないように保存したデータ」のことで，ひとつのファイルは全体としてみれば巨大な2進数を記録したものである。
あるいはそれを8桁づつ区切ったバイトの列を記録したものと考えてもよい。
「8桁づつ区切ったら、何桁か余ってしまうことがあるのでは」と思うかもしれないが、かならず8桁を単位として記録するのが決まりである。

ではここで `hexdump` および `head` というターミナルコマンドを用いて，「吾輩は猫である」のファイルの最初の 160 バイトを見てみよう<sup>注1</sup>：

In [5]:
!hexdump -e '16/1 "%02x " "\n"' neko-noseg.txt | head -10

e4 b8 80 0a e5 90 be e8 bc a9 e3 81 af e7 8c ab
e3 81 a7 e3 81 82 e3 82 8b e3 80 82 0a e5 90 8d
e5 89 8d e3 81 af e3 81 be e3 81 a0 e7 84 a1 e3
81 84 e3 80 82 0a e3 81 a9 e3 81 93 e3 81 a7 e7
94 9f e3 82 8c e3 81 9f e3 81 8b e3 81 a8 e3 82
93 e3 81 a8 e8 a6 8b e5 bd 93 e3 81 8c e3 81 a4
e3 81 8b e3 81 ac e3 80 82 0a e4 bd 95 e3 81 a7
e3 82 82 e8 96 84 e6 9a 97 e3 81 84 e3 81 98 e3
82 81 e3 81 98 e3 82 81 e3 81 97 e3 81 9f e6 89
80 e3 81 a7 e3 83 8b e3 83 a3 e3 83 bc e3 83 8b
hexdump: stdout: Broken pipe


`e4 b8 80 0a e5 90 be e8 bc a9 e3 81 af e7 8c ab` のように並んでいる部分がファイルの中身を1バイトずつ16進表記で表示している。

ファイルが全部で何バイトあるかは、ターミナルコマンド `ls -l ファイル名` で確かめられる：

In [6]:
!ls -l neko-noseg.txt

-rw-r--r--@ 1 kaz  staff  963805 12月 18 16:03 neko-noseg.txt


左から5列目の数値 `963805` が、このファイルは全部で 963,805 バイトであることを表している。

注1: `hexdump` コマンドの後の `-e '16/1 "%02x " "\n"'` の部分は「16バイトを1行として，1バイトずつ16進数で表示せよ」という意味になる．別に覚えたりする必要はない．

## テキストファイルと文字コード
さて上で見たように、ファイル `"neko-noseg.txt"` の先頭の3バイトは `e4 b8 80` だが、これは最初の章の番号である "ー" （いち）の文字を表している。そのことは以下のように確かめられる：

In [7]:
bytes([0xe4, 0xb8, 0x80]).decode("utf-8")

'一'

ここで
* `bytes(...)` は `...` の部分でリストとして渡したバイトの列（各要素は 0 〜 255 の整数）を並べたデータを作る
* `.decode("utf-8")` はバイトの並びを "utf-8" という決まりに従って文字に対応づけた結果を返す
* `0xe4` のような書き方の先頭の `0x` は16進表記の整数であることを表す

また、ファイルの4バイト目の `0a` は改行文字（`\n`）を表すことが以下のように確かめられる：

In [8]:
bytes([0x0a]).decode("utf-8")

'\n'

`bytes` や `decode` の使い方について覚えたりする必要は特にない．「バイトの並びでデータを表している」ということが分かればそれでよい．

#### ミニ練習: バイト列が表す文字
ファイルの5バイト目から7バイト目の `e5 90 be` が utf-8 で何の文字を表すか確認しなさい：

In [9]:
# 確認するプログラムを書きなさい
bytes([0xe5, 0x90, 0xbe]).decode("utf-8")

'吾'

逆にある文字列が "utf-8" の決まりに従ってどのようなバイト列で表されるかは以下のように確かめることができる：

In [13]:
"猫".encode("utf-8")

b'\xe7\x8c\xab'

表示結果の `b'...'` はこれがバイト列であることを表し、`\xe7` の `\x` の部分は16進表示を表す。

よってこの結果は、「猫」という文字は "utf-8" では `e7 8c ab` という3バイトで表されることを意味している。

#### ミニ練習：文字を表すバイト列
「名」の文字が "utf-8" でどのようなバイト列で表されるかを確かめなさい：

In [14]:
# 確かめなさい
"名".encode("utf-8")

b'\xe5\x90\x8d'

さらに、上で `hexdump` を用いて表示した "neko-noseg.txt" のバイト列の中で、
2回目に出てくる改行文字 `0a` の直後（つまり元のテキストファイルの３行目の先頭）が「名」の文字であることを目で見て確認しなさい（「名前はまだない」の「名」）。

---
あるバイトの並びが、どの文字に対応するかを決める "utf-8" のようなルールを**文字コード**という。

単にバイト（整数）の並びではなく、ある文字コードに従った文字の並びを表すファイルを**テキストファイル**という。
Mac 上で Python を使う場合、特に指定しなければテキストファイルを読み書きするときの文字コードは "utf-8" になる。

ファイルには「テキストファイル」以外に**バイナリファイル**と呼ばれるものがある。
バイナリファイルは文字列（テキスト）ではなく、単にバイトが並んだデータで、ファイル中のどのバイト列が何を表すかは特定の決まりに従っている。その決まりのことを**ファイルフォーマット**という。

ファイルフォーマットは例えば `.wav` (音声ファイルの一種）、`.jpg`（画像ファイルの一種）のようにファイル名の最後につける拡張子で表すことが多い。
テキストファイルの拡張子には通常 `.txt` を使う。

このノートブックでは主にテキストファイルの読み書きを練習する。

## テキストファイルの読み込み
### ファイルの open

テキストファイルを読み書きする場合には、まず `open(ファイル名)` によってファイルオブジェクトを作成する。

ファイルオブジェクトというのは、読み込み時にはそこからファイルの中身のデータが流れ出てくる「蛇口（じゃぐち）」のようなもの、
書き込み時にはそこからファイルにデータを注ぎ込む「注入口」のようなものだと思えばよい。

読み込み用にファイルオブジェクトを作成する場合は `ファイル名` だけを引数として `open(ファイル名)` とすればファイルオブジェクトが返される。書き込み用の場合の `open` の引数については後述する。

### 1行読み込む（readline）

読み込み用に作成したファイルオブジェクトに対して `readline()` メソッドを使うと、最初の1行分のテキストが返される：

In [15]:
# "neko-noseg.txt" のファイルオブジェクトを作成する
f = open("neko-noseg.txt") 

# 1行読み込む
line = f.readline()

# 読み込んだ行を表示
print(line)

一



続けてもう一度、引数なしで `readline` メソッドを使うと、次の1行が読み込まれて返される：

In [16]:
# 次の行を読み込む
line = f.readline()

# 読み込んだ行を表示
print(line)

吾輩は猫である。



上のセルをさらに何回か実行し、実行するたびに次の行が読み込まれて表示されることを確かめなさい。

---
`readline` は正確には次のように動作する：
* ファイル上で次の改行文字 `0a` までのバイト列を読み込む。ファイルの最後まで `0a` がなければ、残り全てのバイト列を読み込む。
* 読み込んだバイト列をある文字コード（今の場合は "utf-8"）に従って解釈し、文字列として返す。

「蛇口」の例えで言えば、`readline` は 1. 蛇口を開く、2. 出てきたバイト列を集める、3. バイト `0a` が出てきたら蛇口をいったん閉じる、という動作をする。

細かい点だが、`0a` まで読み込んだ場合、返される文字列の最後には改行文字（`'\n'`）が含まれている。
これは `readline` で返される文字列をセルの最後に直接書くと確認できる：

In [17]:
line = f.readline()
line

'名前はまだ無い。\n'

ファイルの最後まで読み込んで、さらに `readline` を実行するとどうなるだろうか。

`"neko-noseg.txt"` は9000行と少しあるので、とりあえず 10000回 `readline` を実行して最後になにが返されるか確認しよう：

In [18]:
for _ in range(10000): # ファイルの最後まで行っても読み込み続ける
    line = f.readline() # 1行読み込む

# 10000回目に読み込まれたものを表示
line

''

ファイルの終わりまで読み込んで、さらに `readline` を実行すると、空の文字列 `''` が返されることがわかる。

さきほど「細かい点」として触れたように、`readline` は「改行文字まで読み込んで、その改行文字を含めて返す」ので、空の文字列が返されたらファイルの最後まで読み込み終わったことが分かる（なぜでしょう？）。

### ファイルの close
最後までファイルを読み終わったら、「そのファイルオブジェクトはもう使わない」ということを宣言する必要がある。

ファイルオブジェクトを `f` とするとき、この「もう使わない」という宣言は `f.close()` によって実行される。

いらなくなった（最後まで読んだ）ファイルオブジェクトを `close()` しないで、次々に新しいファイルオブジェクトを `open` すると、いつか新しいファイルが開けなくなる。これを避けるために「もう使わないファイルは `close` する」ということを習慣づけよう。

### 1行づつ読んでファイル全体を処理するパターン
以上をまとめると、
* テキストファイルを開き
* 1行づつ読み込んで何かの処理をし
* 最後まで読み終わったら終わる

というよくある処理パターンは以下のように書ける：
```python
f = open(ファイル名) # ファイルを開く

while True:　           # break するまでループし続ける
    line = f.readline() # ファイルオブジェクト f から1行読み込む
    
    if line == '': # 空文字列だったら
        break      # while ループを終える
        
    ...何か処理をする... # 空文字列でなければ何か処理をする
    
f.close() # ループが終わったらファイルを閉じる
```

### 練習13.1: ファイルの行数を数える
ファイル名を表す文字列 `file_name` を受け取り、そのファイルの行数を返す関数 `count_lines(file_name)` を実装しなさい。
ファイルを最後まで読んだときに返される空文字列は<font color="red">1行と数えない</font>ことに注意しなさい：

In [19]:
def count_lines(file_name):
    # 実装しなさい
    f=open(file_name)
    count=0
    while True:
        line=f.readline()
        if line=='':
            break
        else:
            count+=1
    f.close()
    return count

実装できたらテストしなさい：

In [20]:
# 3行からなるテストデータを作る
!echo "hey" > my-test-text-file-3.txt
!echo "hey" >> my-test-text-file-3.txt
!echo "foo" >> my-test-text-file-3.txt

In [21]:
count_lines("my-test-text-file-3.txt") == 3

True

もっと大きなファイル（"neko-noseg.txt"）でもテストしなさい：

In [22]:
count_lines("neko-noseg.txt") == 9210

True

### 練習13.2: 特定の文字列を含む行数を数える
ファイル名を表す文字列 `file_name` と、検索する文字列 `keyword` を受け取り、ファイル中で文字列 `keyword` を含む行を数えて返す関数 `grep_c(file_name, keyword)` を実装しなさい。

同じ行に何回 `keyword` が出てくるかは**関係なく**、あくまで `keyword` を含む**行の数**を数えることに注意しなさい。

ヒント：ある文字列 `line` が、文字列 `k` を部分文字列として含むかどうかは `k in line` で判定できる。「判定できる」とはどういう意味なのかは、セルを作って `"bc" in "abcde"` や `"xy" in "abcde"` を実行して確かめなさい。

In [23]:
def grep_c(file_name, keyword):
    # 実装しなさい
    count=0
    f=open(file_name)
    for _ in range(count_lines(file_name)):
        line = f.readline()
        if keyword in line:
            count+=1
    return count

実行できたらテストしなさい：

In [24]:
print(grep_c("my-test-text-file-3.txt", "hey") == 2)
print(grep_c("my-test-text-file-3.txt", "foo") == 1)
print(grep_c("my-test-text-file-3.txt", "boo") == 0)

True
True
True


「吾輩は猫である」でもテストしなさい：

In [25]:
grep_c("neko-noseg.txt", "吾輩") == 460

True

## テキストファイルの書き込み
テキストファイルに書き込む場合にも、まず `open` によってファイルオブジェクトを作成する。
ただし「書き込む（**w**rite）」ためのファイルオブジェクトであることを表す `"w"` を第２引数に指定する：
```python
f = open(ファイル名, "w") # 書き込むためのファイルオブジェクトを作成
```

書き込む処理自体は、`print` 関数のキーワード引数 `file` に作成したファイルオブジェクトを指定することで実行できる：
```python
print(書き込むデータ, file=f) # ファイルオブジェクト f に対して書き込む
```
「書き込むデータ」の部分は、文字列、整数、浮動小数点数など、`print` できるものであれば何でもよい、また、
```python
print("答えは", 1, "です", file=f)
```
→ `答えは 1 です` と出力される

のように「書き込むデータ」の引数はいくつでも渡せる（空白区切りで出力される）。

書き込みが全て終わったら、`close()` によってファイルオブジェクトを閉じるのは読み込みの場合と同様。

**ミニ練習**: `print` によるファイルへの書き込み

以下のセルを修正し、画面ではなくファイル "my-write-test-file.txt" に
```
abc
def
123
```
と出力せよ（各行の先頭の空白は出力しない）。

In [26]:
f_out = open("my-write-test-file.txt", "w") # 書き込み用にファイルを開く

print("abc") # 修正する：ファイルに1行書き込む
print("def") # 修正する：ファイルに1行書き込む
print("123") # 修正する：ファイルに1行書き込む

f_out.close() # ファイルを閉じる

abc
def
123


上のファイルを修正して実行したら、正しく書き込めているか確認せよ。

`cat ファイル名` は指定したファイルを画面に書き出すターミナルコマンドである：

In [27]:
!cat my-write-test-file.txt

### 練習13.3: 先頭行の書き出し
入力ファイル `in_file_name` の先頭の `n_lines` 行を出力ファイル `out_file_name` に書き込む関数 `head(in_file_name, n_lines, out_file_name)` を実装せよ。

入力ファイル `in_file_name` の行数が `n_lines` 未満の場合は、`in_file_name` の内容を全て書き出した時点で（ファイルオブジェクトを close してから）終了せよ。

引数 `in_file_name` および `out_file_name` はファイル名を表す文字列であって、ファイルオブジェクトではないことに注意せよ（つまり関数の中で自分で `open` してファイルオブジェクトを作成する）。

また、`readline` で読み込んだ文字列の最後には改行文字 `\n` が付いていることに注意せよ。このため、読み込んだ文字列 `line` を
```python
print(line, file=出力ファイルオブジェクト)
```
のように `print` すると、`line` の後にさらに改行が出力されるため、結果は「1行おき」のような状態になる。
これを防ぐには、`print` の後に改行を出力しないことを
```python
print(line, file=出力ファイルオブジェクト, end="")
```
のように `end` キーワード引数で指定せよ。
`end` 引数は全ての引数を表示した後に出力される文字列を指定する。
`end` を指定しない時は `end='\n'` を指定したのと同じことになり、改行が出力される。

In [28]:
# 引数：
#   in_file_name: 入力ファイル名（文字列）
#   n_lines: 書き出す行数（整数）
#   out_file_name: 出力ファイル名（文字列）
def head(in_file_name, n_lines, out_file_name):
    # *** 実装しなさい ***
    f=open(in_file_name)
    f_out=open(out_file_name,"w")
    for _ in range(n_lines):
        line=f.readline()
        if line=='':
            break
        else:
            print(line,file=f_out,end="")
    f.close()
    f_out.close()
    return f_out

実装できたらテストしなさい。
以下のテストは関数 `head(in_file_name, n_lines, out_file_name)` と同じことを行うターミナルコマンド
```sh
head -{n_lines} {in_file_name} > {out_file_name}
```
の実行結果と、上で実装した関数の出力結果を、ファイルの相違を調べるターミナルコマンド `diff` でチェックする。
最後に `OK` と表示されればテストは成功、`FAIL` と表示されたらテストは失敗である。

ファイル内容に相違がある場合は
```
< 出力に足りない行
> 出力で余計な行
```
のような形で差分が表示される。

もしも失敗した場合は、ファイル `my-test-text-file-answer.txt`（正解）および `my-test-text-file-output.txt`（プログラムの出力）をエディタなどで開いてみて、どこが違うか確かめよ。

In [29]:
#「正解」のファイルを作る
!head -2 my-test-text-file-3.txt > my-test-text-file-answer.txt

In [30]:
# 実装した関数を実行：先頭2行を書き出す
head("my-test-text-file-3.txt", 2, "my-test-text-file-output.txt")

<_io.TextIOWrapper name='my-test-text-file-output.txt' mode='w' encoding='UTF-8'>

In [31]:
#「正解」と実行結果を比較する
!diff my-test-text-file-answer.txt my-test-text-file-output.txt; if [ "$?" = "0" ]; then echo "OK"; else echo "FAIL"; fi

OK


「吾輩は猫である」でもテストせよ：

In [32]:
#「正解」のファイルを作る
!head -123 neko-noseg.txt > test-head-neko-answer.txt

In [33]:
# 実装した関数を実行：先頭123行を書き出す
head("neko-noseg.txt", 123, "test-head-neko-output.txt")

<_io.TextIOWrapper name='test-head-neko-output.txt' mode='w' encoding='UTF-8'>

In [34]:
#「正解」と実行結果を比較する
!diff test-head-neko-answer.txt test-head-neko-output.txt; if [ "$?" = "0" ]; then echo "OK"; else echo "FAIL"; fi

OK


### 練習13.4: キーワードを含む行の書き出し
入力ファイル `in_file_name` のうち、 `keyword` を含む行を全て出力ファイル `out_file_name` に書き込む関数 `grep(in_file_name, keyword, out_file_name)` を実装せよ。

ヒント（ダブルクリックで表示）
<!--
おおむね以下のようにすればよい

* 入力ファイルを open 関数で開く
* 出力ファイルを open 関数で開く（第2引数は "w"）
* while True:  # 無限ループ
    * 入力ファイルを1行読み込む
    * 読み込んだ行が空行（''）だったら終了（break）
    * 読み込んだ行に keyword が含まれていたら出力ファイルに書き出す
* 入力ファイル・出力ファイルを閉じる
-->

In [35]:
# 引数：
#   in_file_name: 入力ファイル名（文字列）
#   keyword: 検索するキーワード（文字列）
#   out_file_name: 出力ファイル名（文字列）
def grep(in_file_name, keyword, out_file_name):
    # *** 実装しなさい ***
    f=open(in_file_name)
    f_out=open(out_file_name,"w")
    while True:
        line=f.readline()
        if line=='':
            break
        elif keyword in line:
            print(line,file=f_out,end="")
    f.close()
    f_out.close()
    return f_out    

以下のセルを実行すると「吾輩は猫である」から「吾輩」を含む行が "my-grep-test-output.txt" に書き出されるはずである。

In [36]:
grep("neko-noseg.txt", "吾輩", "test-grep-neko-output.txt")

<_io.TextIOWrapper name='test-grep-neko-output.txt' mode='w' encoding='UTF-8'>

まず結果が正しそうか、出力ファイルの先頭5行を表示してみなさい：

In [37]:
!head -5 test-grep-neko-output.txt

吾輩は猫である。
吾輩はここで始めて人間というものを見た。
吾輩は藁の上から急に笹原の中へ棄てられたのである。
吾輩は池の前に坐ってどうしたらよかろうと考えて見た。
縁は不思議なもので、もしこの竹垣が破れていなかったなら、吾輩はついに路傍に餓死したかも知れんのである。


上で実装した関数 `grep(in_file_name, keyword, out_file_name)` と同じ動作をするターミナルコマンドは
```sh
grep {keyword} {in_file_name} > {out_file_name}
```
である。この結果と比べてテストしなさい：

In [38]:
#「正解」を作る
!grep 吾輩 neko-noseg.txt > test-grep-neko-answer.txt

# 実装した関数の結果と比較
!diff test-grep-neko-answer.txt test-grep-neko-output.txt; if [ "$?" = "0" ]; then echo "OK"; else echo "FAIL"; fi

OK


---
以上で今週の課題は終わりです。

- Run -> Run All Cells を実行して保存のボタンを押し、<font color="red">全ての実行結果を保存した上で</font>提出してください。
- 実行結果が保存されていない問題は<font color="red">採点しません</font>